# 🏦 Bank Cash Optimization — Phase 4: Model Training

---

**Project:** Bank Cash Optimization  
**Notebook:** 04_Model_Training  
**Author:** Abdul Qadeer  

## 🎯 Objective

Train and evaluate ML models to predict **daily cash withdrawal per branch**.  

**This notebook covers:**
1. Load prepared train/test data
2. Baseline model (naive reference)
3. XGBoost model training
4. LightGBM model training
5. Hyperparameter tuning (best model)
6. Evaluation: MAE, RMSE, MAPE, R²
7. Actual vs Predicted visualization
8. Feature importance plot
9. Save best model


---
## Step 1 — Import Libraries

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import joblib, json, os

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.dummy import DummyRegressor
import xgboost as xgb
import lightgbm as lgb
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

pd.set_option('display.float_format', '{:,.2f}'.format)
os.makedirs('models',    exist_ok=True)
os.makedirs('eda_plots', exist_ok=True)

plt.rcParams.update({
    'figure.dpi'       : 130,
    'axes.titlesize'   : 13,
    'axes.titleweight' : 'bold',
    'axes.labelsize'   : 11,
    'axes.spines.top'  : False,
    'axes.spines.right': False,
    'figure.facecolor' : 'white',
    'axes.facecolor'   : '#F9FAFB',
    'axes.grid'        : True,
    'grid.color'       : '#E5E7EB',
})

print('✅ Libraries imported successfully')

✅ Libraries imported successfully


---
## Step 2 — Load Prepared Data

In [2]:
X_train = pd.read_csv('model_data/X_train.csv')
X_test  = pd.read_csv('model_data/X_test.csv')
y_train = pd.read_csv('model_data/y_train.csv').squeeze()
y_test  = pd.read_csv('model_data/y_test.csv').squeeze()

with open('model_data/feature_cols.json') as f:
    feature_cols = json.load(f)

print(f'✅ Data loaded successfully')
print(f'   X_train : {X_train.shape}  |  y_train : {y_train.shape}')
print(f'   X_test  : {X_test.shape}   |  y_test  : {y_test.shape}')
print(f'   Features: {len(feature_cols)}')
print(f'\n   Target Stats (y_train):')
print(f'   Mean   : PKR {y_train.mean()/1e6:,.1f} Million')
print(f'   Median : PKR {y_train.median()/1e6:,.1f} Million')
print(f'   Max    : PKR {y_train.max()/1e6:,.1f} Million')
print(f'   Min    : PKR {y_train.min()/1e6:,.1f} Million')

✅ Data loaded successfully
   X_train : (6851, 22)  |  y_train : (6851,)
   X_test  : (1731, 22)   |  y_test  : (1731,)
   Features: 22

   Target Stats (y_train):
   Mean   : PKR 46.4 Million
   Median : PKR 35.4 Million
   Max    : PKR 320.7 Million
   Min    : PKR 0.0 Million


---
## Step 3 — Helper: Evaluation Function

Ek function jo MAE, RMSE, MAPE, R² sab ek saath calculate kare.

In [3]:
def evaluate_model(name, y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    # MAPE: only on non-zero actuals (avoids division by near-zero)
    mask = np.array(y_true) > 0
    mape = np.mean(np.abs((np.array(y_true)[mask] - np.array(y_pred)[mask]) / np.array(y_true)[mask])) * 100
    r2   = r2_score(y_true, y_pred)
    print(f'\n📊 {name} — Evaluation Results:')
    print(f'   MAE  : PKR {mae/1e6:>8,.2f} Million   (Average error per prediction)')
    print(f'   RMSE : PKR {rmse/1e6:>8,.2f} Million   (Penalizes big errors more)')
    print(f'   MAPE :     {mape:>8,.2f}%          (% error on average)')
    print(f'   R²   :     {r2:>8,.4f}            (1.0 = perfect, >0.85 = good)')
    return {'Model': name, 'MAE': mae, 'RMSE': rmse, 'MAPE': mape, 'R2': r2}

results = []  # Store all model results for comparison
print('✅ Evaluation function ready')

✅ Evaluation function ready


---
## Step 4 — Baseline Model (Naive Reference)

Baseline = sabse simple prediction.  
Hamare smart model ko is se behtar hona zaroori hai.

In [4]:
# Baseline: predict the mean of training target for everything
baseline = DummyRegressor(strategy='mean')
baseline.fit(X_train, y_train)
y_pred_baseline = baseline.predict(X_test)

res = evaluate_model('Baseline (Mean Predictor)', y_test, y_pred_baseline)
results.append(res)
print('\n   This is our minimum bar — our real model MUST beat this.')


📊 Baseline (Mean Predictor) — Evaluation Results:
   MAE  : PKR    21.85 Million   (Average error per prediction)
   RMSE : PKR    30.62 Million   (Penalizes big errors more)
   MAPE :       202.89%          (% error on average)
   R²   :      -0.0005            (1.0 = perfect, >0.85 = good)

   This is our minimum bar — our real model MUST beat this.


---
## Step 5 — XGBoost Model

XGBoost (Extreme Gradient Boosting) — tree-based ensemble model, best for tabular data.

In [5]:
xgb_model = XGBRegressor(
    n_estimators      = 500,
    learning_rate     = 0.05,
    max_depth         = 6,
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    min_child_weight  = 3,
    reg_alpha         = 0.1,
    reg_lambda        = 1.0,
    random_state      = 42,
    n_jobs            = -1,
    verbosity         = 0,
)

xgb_model.fit(
    X_train, y_train,
    eval_set        = [(X_test, y_test)],
    verbose         = False
)

y_pred_xgb = xgb_model.predict(X_test)
res = evaluate_model('XGBoost', y_test, y_pred_xgb)
results.append(res)
print('\n\u2705 XGBoost training complete')


📊 XGBoost — Evaluation Results:
   MAE  : PKR    14.25 Million   (Average error per prediction)
   RMSE : PKR    20.29 Million   (Penalizes big errors more)
   MAPE :        62.13%          (% error on average)
   R²   :       0.5607            (1.0 = perfect, >0.85 = good)

✅ XGBoost training complete


---
## Step 6 — LightGBM Model

LightGBM — faster than XGBoost, often comparable or better accuracy.

In [6]:
lgb_model = LGBMRegressor(
    n_estimators      = 500,
    learning_rate     = 0.05,
    max_depth         = 6,
    num_leaves        = 63,
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    min_child_samples = 20,
    reg_alpha         = 0.1,
    reg_lambda        = 1.0,
    random_state      = 42,
    n_jobs            = -1,
    verbose           = -1,
)

lgb_model.fit(
    X_train, y_train,
    eval_set = [(X_test, y_test)],
)

y_pred_lgb = lgb_model.predict(X_test)
res = evaluate_model('LightGBM', y_test, y_pred_lgb)
results.append(res)


📊 LightGBM — Evaluation Results:
   MAE  : PKR    14.15 Million   (Average error per prediction)
   RMSE : PKR    20.30 Million   (Penalizes big errors more)
   MAPE :        58.47%          (% error on average)
   R²   :       0.5600            (1.0 = perfect, >0.85 = good)


---
## Step 7 — Model Comparison Table

In [7]:
results_df = pd.DataFrame(results)
results_df['MAE_M']  = results_df['MAE']  / 1e6
results_df['RMSE_M'] = results_df['RMSE'] / 1e6

print('\n' + '='*65)
print('MODEL COMPARISON SUMMARY')
print('='*65)
display_df = results_df[['Model','MAE_M','RMSE_M','MAPE','R2']].copy()
display_df.columns = ['Model', 'MAE (Million PKR)', 'RMSE (Million PKR)', 'MAPE (%)', 'R²']
print(display_df.to_string(index=False))
print('='*65)

# Best model based on R2
best_row = results_df.loc[results_df['R2'].idxmax()]
print(f'\n🏆 Best Model: {best_row["Model"]}  (R² = {best_row["R2"]:.4f})')


MODEL COMPARISON SUMMARY
                    Model  MAE (Million PKR)  RMSE (Million PKR)  MAPE (%)    R²
Baseline (Mean Predictor)              21.85               30.62    202.89 -0.00
                  XGBoost              14.25               20.29     62.13  0.56
                 LightGBM              14.15               20.30     58.47  0.56

🏆 Best Model: XGBoost  (R² = 0.5607)


---
## Step 8 — Actual vs Predicted Plot

In [8]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle('Actual vs Predicted — Daily Cash Withdrawal', fontsize=14, fontweight='bold')

for ax, (name, y_pred) in zip(axes, [('XGBoost', y_pred_xgb), ('LightGBM', y_pred_lgb)]):
    r2 = r2_score(y_test, y_pred)
    ax.scatter(y_test/1e6, y_pred/1e6, alpha=0.3, color='#1F4E79', s=15, label='Predictions')
    mn = min(y_test.min(), y_pred.min()) / 1e6
    mx = max(y_test.max(), y_pred.max()) / 1e6
    ax.plot([mn, mx], [mn, mx], color='#D6462B', linewidth=2, linestyle='--', label='Perfect Prediction')
    ax.set_title(f'{name}  (R² = {r2:.4f})')
    ax.set_xlabel('Actual (Million PKR)')
    ax.set_ylabel('Predicted (Million PKR)')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('eda_plots/10_actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.close()
print('✅ Actual vs Predicted plot saved → eda_plots/10_actual_vs_predicted.png')

✅ Actual vs Predicted plot saved → eda_plots/10_actual_vs_predicted.png


---
## Step 9 — Prediction Timeline Plot (Sample Branch)

In [9]:
# Load full daily data to get dates for test set
daily_full = pd.read_csv('model_data/daily_full.csv')
daily_full['start_date'] = pd.to_datetime(daily_full['start_date'])
daily_full = daily_full.sort_values('start_date').reset_index(drop=True)

cutoff_idx  = int(len(daily_full) * 0.80)
cutoff_date = daily_full.iloc[cutoff_idx]['start_date']
test_full   = daily_full[daily_full['start_date'] >= cutoff_date].reset_index(drop=True)

# Pick most active branch
top_branch = daily_full.groupby('tran_br_code')['Daily_Total_Debit'].sum().idxmax()
mask_test  = test_full['tran_br_code'] == top_branch

branch_dates  = test_full.loc[mask_test, 'start_date'].values
branch_actual = y_test.values[mask_test.values]
branch_xgb    = y_pred_xgb[mask_test.values]
branch_lgb    = y_pred_lgb[mask_test.values]

fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(branch_dates, branch_actual/1e6, color='black',   linewidth=2,   label='Actual',   zorder=3)
ax.plot(branch_dates, branch_xgb/1e6,   color='#1F4E79', linewidth=1.5, linestyle='--', label='XGBoost Predicted')
ax.plot(branch_dates, branch_lgb/1e6,   color='#D6462B', linewidth=1.5, linestyle=':',  label='LightGBM Predicted')
ax.set_title(f'Branch {top_branch} — Actual vs Predicted Daily Cash Withdrawal (Test Period)', fontsize=13, fontweight='bold')
ax.set_xlabel('Date'); ax.set_ylabel('Cash Withdrawal (Million PKR)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{v:.0f}M'))
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('eda_plots/11_prediction_timeline.png', dpi=150, bbox_inches='tight')
plt.close()
print(f'✅ Timeline plot saved → eda_plots/11_prediction_timeline.png  (Branch: {top_branch})')

✅ Timeline plot saved → eda_plots/11_prediction_timeline.png  (Branch: 104)


---
## Step 10 — Feature Importance Plot

In [10]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('Feature Importance — Which Features Matter Most?', fontsize=14, fontweight='bold')

# XGBoost importance
xgb_imp = pd.Series(xgb_model.feature_importances_, index=feature_cols).sort_values(ascending=True)
xgb_imp.plot(kind='barh', ax=axes[0], color='#1F4E79', edgecolor='white')
axes[0].set_title('XGBoost — Feature Importance')
axes[0].set_xlabel('Importance Score')

# LightGBM importance
lgb_imp = pd.Series(lgb_model.feature_importances_, index=feature_cols).sort_values(ascending=True)
lgb_imp.plot(kind='barh', ax=axes[1], color='#D6462B', edgecolor='white')
axes[1].set_title('LightGBM — Feature Importance')
axes[1].set_xlabel('Importance Score')

plt.tight_layout()
plt.savefig('eda_plots/12_feature_importance.png', dpi=150, bbox_inches='tight')
plt.close()
print('✅ Feature importance plot saved → eda_plots/12_feature_importance.png')

✅ Feature importance plot saved → eda_plots/12_feature_importance.png


---
## Step 11 — Save Best Model

In [11]:
# Determine best model based on R2
r2_xgb = r2_score(y_test, y_pred_xgb)
r2_lgb = r2_score(y_test, y_pred_lgb)

if r2_xgb >= r2_lgb:
    best_model      = xgb_model
    best_model_name = 'XGBoost'
    best_r2         = r2_xgb
    best_pred       = y_pred_xgb
else:
    best_model      = lgb_model
    best_model_name = 'LightGBM'
    best_r2         = r2_lgb
    best_pred       = y_pred_lgb

# Save best model
joblib.dump(best_model, 'models/best_model.pkl')
joblib.dump(xgb_model,  'models/xgb_model.pkl')
joblib.dump(lgb_model,  'models/lgb_model.pkl')

# Save results summary
results_df.to_csv('models/model_results.csv', index=False)

print(f'✅ Models saved in models/ folder:')
print(f'   📄 best_model.pkl  — {best_model_name} (R² = {best_r2:.4f})')
print(f'   📄 xgb_model.pkl   — XGBoost')
print(f'   📄 lgb_model.pkl   — LightGBM')
print(f'   📄 model_results.csv')
print(f'\n🎉 Phase 4 COMPLETE!')
print(f'   ➡️  Next: Phase 5 — XAI (Explainable AI with SHAP)')

✅ Models saved in models/ folder:
   📄 best_model.pkl  — XGBoost (R² = 0.5607)
   📄 xgb_model.pkl   — XGBoost
   📄 lgb_model.pkl   — LightGBM
   📄 model_results.csv

🎉 Phase 4 COMPLETE!
   ➡️  Next: Phase 5 — XAI (Explainable AI with SHAP)


---
## ✅ Phase 4 Summary

| Item | Detail |
|---|---|
| **Models Trained** | Baseline, XGBoost, LightGBM |
| **Best Model** | See Step 11 output |
| **Plots Saved** | 10_actual_vs_predicted, 11_prediction_timeline, 12_feature_importance |
| **Models Saved** | models/best_model.pkl, xgb_model.pkl, lgb_model.pkl |

### ➡️ Next: Phase 5 — XAI (SHAP Explainability)